# Flow API Platform - Google Colab (12 GB RAM - 100% Gratis)
Menjalankan Flow API Platform di cloud Google secara otomatis tanpa kartu kredit dan tanpa menyalakan Mac Anda.

### Cara Pakai:
1. Klik tombol **Play** (Jalankan cell) di bawah.
2. Tunggu 1–2 menit sampai muncul link **https://xxx.trycloudflare.com**.
3. Buka link Swagger UI tersebut, lalu gunakan endpoint `/v1/cookies/quick-login` untuk menyuntikkan cookies Flow Anda!

In [ ]:
# 1. Install Node.js 22.x (Wajib Node 22.5+ untuk DatabaseSync SQLite)
print("📦 Menginstall Node.js 22 & dependensi Linux...")
!curl -fsSL https://deb.nodesource.com/setup_22.x | bash - > /dev/null 2>&1
!apt-get install -y nodejs ffmpeg chromium-browser > /dev/null 2>&1

# 2. Install Cloudflare Tunnel (cloudflared)
print("🌐 Menginstall Cloudflare Tunnel...")
!curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -o cloudflared.deb
!dpkg -i cloudflared.deb > /dev/null 2>&1
!rm -f cloudflared.deb

# 3. Clone Repository terbaru
print("📥 Mengambil source code Flow API Platform...")
!rm -rf flow-api-platform
!git clone https://github.com/reiulen/flow-api-platform.git
%cd flow-api-platform

# 4. Install Dependencies & Playwright Browser
print("⚙️ Memasang dependensi Node.js & Playwright Chromium...")
!npm install --omit=dev
!npx playwright install chromium

# 5. Jalankan Server & Cloudflare Tunnel
import subprocess, time, re, os

print("\n🚀 Menjalankan Flow API Server...")
server_env = os.environ.copy()
server_env.update({
    "HOST": "127.0.0.1",
    "PORT": "8787",
    "FLOW_HEADLESS": "true",
    "FLOW_EMBEDDED_WORKER": "true",
    "FLOW_API_KEY": "6107412d0c24d8144d7cdb847c340e19",
    "FLOW_MASTER_KEY": "F11fjnHM12v8D506DV2m8TdTrbSESdtccXehYbA/Fis="
})

server_proc = subprocess.Popen(["node", "src/server.js"], env=server_env)
time.sleep(3)

print("🌐 Menghubungkan Cloudflare Tunnel publik...")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8787"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

for line in tunnel_proc.stdout:
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        url = match.group(0)
        print("\n" + "=" * 65)
        print("🎉 FLOW API BERHASIL AKTIF DI GOOGLE CLOUD (12 GB RAM)!")
        print(f"👉 Link Swagger UI : {url}/docs")
        print(f"👉 Link Scalar Ref : {url}/scalar")
        print(f"👉 API Endpoint    : {url}/v1")
        print(f"👉 API Key         : 6107412d0c24d8144d7cdb847c340e19")
        print("=" * 65 + "\n")
        break

try:
    tunnel_proc.wait()
except KeyboardInterrupt:
    server_proc.terminate()
    tunnel_proc.terminate()
